In [1]:
import sqlite3
import pandas as pd
import matplotlib.pyplot as plt

class InboundAnalyzer:
    def __init__(self, db_path):
        self.db_path = db_path

    def _query(self, sql):
        conn = sqlite3.connect(self.db_path)
        df = pd.read_sql_query(sql, conn)
        conn.close()
        return df

    def get_yearly_totals(self):
        sql = """
        SELECT country, year, SUM(visitors) AS total
        FROM inbound_country
        GROUP BY country, year;
        """
        return self._query(sql)

    def get_recovery_rate(self, base_year=2019):
        sql = f"""
        WITH yearly AS (
            SELECT country, year, SUM(visitors) AS total
            FROM inbound_country
            GROUP BY country, year
        ),
        latest AS (
            SELECT MAX(year) AS y FROM inbound_country
        )
        SELECT
            y2019.country,
            y2019.total AS base_total,
            ylatest.total AS latest_total,
            1.0 * ylatest.total / y2019.total AS recovery_rate
        FROM yearly y2019
        JOIN latest ON 1=1
        JOIN yearly ylatest
          ON ylatest.country = y2019.country
         AND ylatest.year = latest.y
        WHERE y2019.year = {base_year}
          AND y2019.total > 0
        ORDER BY recovery_rate DESC;
        """
        return self._query(sql)

    def plot_recovery(self, base_year=2019, top_n=10):
        df = self.get_recovery_rate(base_year).head(top_n)

        plt.figure(figsize=(10,5))
        plt.bar(df["country"], df["recovery_rate"])
        plt.xticks(rotation=45)
        plt.title(f"Recovery Rate from {base_year}")
        plt.ylabel("Recovery Rate")
        plt.tight_layout()
        plt.show()
